# 003_meeting_semantic_lens — 業務語意鏡頭表＋候選 View（DDL）

- 用途：方向 C（規格 §8.3）的第一步。建語意鏡頭表 `g_meeting_items_sem_bu` 與兩個 View：`v_meeting_items_sem_cand`（候選規則用 SQL 先算、給 Stage A 當輸入）、`v_meeting_items_sem_q`（鏡頭表消費介面）。只建結構、不灌資料；資料由之後的 Stage A job 寫入。
- **正本是 [c04] `table_specs` 與 [c05] `view_specs`**。改欄 = 開新流水號 migration 做 `ALTER TABLE ADD COLUMNS`（§2.3 只增不改）；View 無狀態，改 [c05] 重跑即可。
- 鏡頭表**一列一個 `item_key`**（不是 `item_id`）：中立層重萃後 `item_id` 會變、`item_key` 不變，PM 的 `verified` / `review_note` 跟著 `item_key` 留下，Stage A MERGE 不覆蓋這兩欄。
- 候選規則（§8.3，Stage A 的先驗）：`statement_type = definition` 或「`terms_mentioned` 非空且 quote 含 指 / 定義 / 算法 / 口徑」→ `semantic`；`statement_type ∈ {opinion, fact}` 且 `topic_tag_primary ∈ {demand, commercial_relation, supply_chain, capacity}` → `intel`；其餘 `event`。先看 `v_meeting_items_sem_cand` 的分布再決定 LLM 只判哪些。
- 冪等：表 `CREATE TABLE IF NOT EXISTS`；View `CREATE OR REPLACE VIEW`。前置：002 的四張表要存在（View 引用 `g_meeting_items_bu`、`s_meeting_record_clean_bu`）。
- 參數（widgets）：`catalog`、`schema`、`dry_run`（預設 `true`：只驗證 spec 與印出 DDL，不建任何物件）
- 負責人 / 更新日期：（填）/ 2026-10-19

| 物件 | 鍵 / 來源 | 內容 | 敏感度 |
|---|---|---|---|
| `g_meeting_items_sem_bu` | `item_key` | §8.3 YAML items 區轉 Delta：category / term / summary / quote / date / source / tags / as_of / dup_of / route_hint / extract_note ＋ PM 的 verified / review_note ＋ 先驗 category_rule ＋ lineage | S0–S2 |
| `v_meeting_items_sem_cand` | items ⋈ clean ⋈ sem | 中立層有效條目＋`category_rule` / `rule_basis` / `week` / `source` / `tags` / `weight_hint`＋`already_classified` | S2 |
| `v_meeting_items_sem_q` | sem | 只留 `active`，陣列欄字串版，`rule_disagree` 抽審旗標 | S2 |
| `b_meeting_migration_log` | `(slug, run_at)` | 執行紀錄（IF NOT EXISTS） | – |

**不在本 migration**：Stage A job（LLM 判類、寫鏡頭表）；Sales Brief 鏡頭兩張表與 View（§3.4、§3.5 → 004）；column mask / GRANT；分應用 View。


In [ ]:
# [c01] params
dbutils.widgets.text("catalog", "micenter")
dbutils.widgets.text("schema", "mi3_datahub_prod")
dbutils.widgets.dropdown("dry_run", "true", ["true", "false"])

catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()
dry_run = dbutils.widgets.get("dry_run") == "true"
assert catalog and schema, "catalog / schema 不可為空"
print({"catalog": catalog, "schema": schema, "dry_run": dry_run})

In [ ]:
# [c02] imports
# 本 cell 與 [c03]～[c05] 不碰 spark / dbutils，可被 tests/ 載入。
import json
import re

SLUG = "003_meeting_semantic_lens"
UNIT = "bu"  # 單位代碼：表名後綴與 unit 欄預設值（規格 §3 命名規則）
COL_RE = r"^[a-z][a-z0-9_]*$"
TYPE_RE = r"^(STRING|BIGINT|INT|DOUBLE|BOOLEAN|DATE|TIMESTAMP|ARRAY<STRING>)$"

# 語意鏡頭表的承接鍵一定要有（§8.3：PM 審核欄以 item_key 承接；重萃後 item_id 會變、item_key 不變）。
REQUIRED_COLS = {
    "g_meeting_items_sem_bu": ["item_key", "item_id", "category", "category_rule", "verified", "review_note"],
}
# S3 欄位：只准落在 s_meeting_record_mail_bu / s_meeting_participants_bu；任何 View 不得暴露。
S3_COLS = {
    "display_name_raw", "name_zh", "name_en", "email", "emp_id", "account",
    "forwarder_name_zh", "forwarder_emp_id", "forwarder_account",
    "mail_to_raw", "mail_cc_raw", "mail_bcc_raw", "body_display",
}

# 候選規則（規格 §8.3「Stage A 的先驗，仍由 LLM 判」）。SQL 先算分布，再決定 LLM 只判哪些。
CATEGORIES = ("semantic", "intel", "event")
INTEL_STATEMENT_TYPES = ("opinion", "fact")
# 2026-10-19 D30：加 price / cost / allocation（探索 SQL 看到 event 裡 fact/cost 100、fact/allocation 100、fact/price 74 多是商務情報）
INTEL_TAGS = ("demand", "commercial_relation", "supply_chain", "capacity", "price", "cost", "allocation")
SEMANTIC_HINT_RE = "指|定義|算法|口徑"  # quote 含這些字且 terms_mentioned 非空 → semantic 候選（Spark RLIKE）
LOW_WEIGHT_CONTENT_TYPES = ("inline_minutes",)  # §8.3 紀律調整：已書面化的紀要降權（實測占 79%，只當欄位不當門檻）
DUP_MIN_CHARS = 20  # quote 正規化後至少這麼長才做同文重複比對（dup_key）；太短的句子（OK、同意）不算重複


In [ ]:
# [c03] pure_ddl
# spec → SQL 字串的純函式；不碰 spark。測試在本機用同一份 spec 建空表跑 View 的 SELECT。
def q(s: str | None) -> str:
    """SQL 單引號字串常值；None → ''。"""
    return "'" + (s or "").replace("\\", "\\\\").replace("'", "\\'") + "'"


def build_create_table(fq: str, spec: dict) -> str:
    """CREATE TABLE IF NOT EXISTS；鍵欄 NOT NULL；有 cluster_by 才加 CLUSTER BY（liquid clustering）。"""
    keys = set(spec["key"])
    cols = ",\n".join(
        f"    {name} {typ}{' NOT NULL' if name in keys else ''} COMMENT {q(comment)}"
        for name, typ, comment in spec["columns"]
    )
    cluster = f"\nCLUSTER BY ({', '.join(spec['cluster_by'])})" if spec.get("cluster_by") else ""
    return f"CREATE TABLE IF NOT EXISTS {fq} (\n{cols}\n) USING DELTA{cluster}\nCOMMENT {q(spec['comment'])}"


def build_select(spec: dict, fq) -> str:
    """View 的查詢本體：spec['select'] = [(expr, alias, comment)]，spec['body'](fq) 回傳 FROM 起的 SQL。
    fq(table_name) -> 可查詢的名稱；正式是三段式，測試用 temp view 名。"""
    select = ",\n".join(f"    {expr} AS {alias}" for expr, alias, _ in spec["select"])
    return f"SELECT\n{select}\n{spec['body'](fq)}"


def build_create_view(fq_view: str, spec: dict, fq) -> str:
    """CREATE OR REPLACE VIEW，欄位清單帶 COMMENT（Genie / 人看的語意）。"""
    col_list = ",\n".join(f"    {alias} COMMENT {q(comment)}" for _, alias, comment in spec["select"])
    return (
        f"CREATE OR REPLACE VIEW {fq_view} (\n{col_list}\n)\nCOMMENT {q(spec['comment'])}\nAS\n"
        + build_select(spec, fq)
    )


def schema_ddl(spec: dict) -> str:
    """欄位 → 'a BIGINT, b STRING'（測試建空表、[c30] 比對型別用）。"""
    return ", ".join(f"{n} {t}" for n, t, _ in spec["columns"])


def add_columns_sql(fq: str, spec: dict, missing: list[str]) -> str:
    """[c30] 發現缺欄時印出的補欄語句（不自動執行；要補就開新流水號 migration）。"""
    by_name = {n: (t, c) for n, t, c in spec["columns"]}
    parts = ", ".join(f"{n} {by_name[n][0]} COMMENT {q(by_name[n][1])}" for n in missing)
    return f"ALTER TABLE {fq} ADD COLUMNS ({parts})"


def validate_specs(tables: dict, views: dict) -> list[str]:
    """回傳錯誤訊息；空 list = 通過。"""
    errs = []
    for t, spec in tables.items():
        names = [c[0] for c in spec["columns"]]
        dup = sorted({n for n in names if names.count(n) > 1})
        if dup:
            errs.append(f"{t} 欄位重複：{dup}")
        for n, typ, comment in spec["columns"]:
            if not re.match(COL_RE, n):
                errs.append(f"{t}.{n} 欄名格式不符（小寫英數底線）")
            if not re.match(TYPE_RE, typ):
                errs.append(f"{t}.{n} 型別不在允許清單：{typ}")
            if not comment:
                errs.append(f"{t}.{n} 沒有 COMMENT")
        for k in list(spec["key"]) + list(spec.get("cluster_by", [])):
            if k not in names:
                errs.append(f"{t} key / cluster_by 欄 {k} 不存在")
        for r in REQUIRED_COLS.get(t, []):
            if r not in names:
                errs.append(f"{t} 缺必要欄 {r}")
        if not spec.get("comment"):
            errs.append(f"{t} 沒有表 COMMENT")
    for v, spec in views.items():
        aliases = [a for _, a, _ in spec["select"]]
        dup = sorted({a for a in aliases if aliases.count(a) > 1})
        if dup:
            errs.append(f"{v} 輸出欄重複：{dup}")
        leak = sorted(S3_COLS & set(aliases))
        if leak:
            errs.append(f"{v} 暴露 S3 欄：{leak}")
        for _, a, c in spec["select"]:
            if not re.match(COL_RE, a):
                errs.append(f"{v}.{a} 欄名格式不符")
            if not c:
                errs.append(f"{v}.{a} 沒有 COMMENT")
        if not spec.get("comment"):
            errs.append(f"{v} 沒有 View COMMENT")
    return errs

In [ ]:
# [c04] table_specs
# 正本。欄位 = (名稱, 型別, COMMENT)。鍵欄自動 NOT NULL。
# g_meeting_items_sem_bu = 規格 §8.3 草案（_prompt_weekly_extract.md v1.1 的 YAML items 區轉 Delta）＋ 回溯 / lineage 欄。
# 一列 = 一個 item_key（不是 item_id）：重萃後同一句話 item_id 會變、item_key 不變，PM 的 verified / review_note 跟著 item_key 留下。
# CLUSTER BY (week)：Stage A 每週寫一批、PM 按週審、Stage B 按週彙整。
_YAML = "同 _prompt_weekly_extract.md YAML"

TABLES = {
    "g_meeting_items_sem_bu": {
        "comment": "會議條目的業務語意鏡頭（方向 C，Stage A 產出）：一列一個 item_key，category = semantic / intel / event，PM 填 verified。中立層欄位複製一份方便審核，正本仍是 g_meeting_items_bu。不含姓名 email。正本：003_meeting_semantic_lens [c04]。",
        "key": ["item_key"],
        "cluster_by": ["week"],
        "columns": [
            ("item_key", "STRING", "承接鍵（= g_meeting_items_bu.item_key）；PM 審核欄跟著它，重萃不變"),
            ("item_id", "STRING", "目前對應的中立條目 id；中立層重萃後由 Stage A 更新"),
            ("pk_seq", "BIGINT", "回溯：來源信"),
            ("segment_no", "INT", "回溯：段序"),
            ("sub_no", "INT", "回溯：子段序"),
            ("unit", "STRING", "單位代碼（本版 bu）"),
            ("week", "STRING", "YYYY-Www（ISO 週），由中立層 extracted_at 推（§8.3）；Stage B 按週彙整"),
            ("category", "STRING", "semantic / intel / event（Stage A 判；" + _YAML + " category）"),
            ("category_rule", "STRING", "SQL 先驗候選（v_meeting_items_sem_cand.category_rule），Stage A 的輸入；與 category 不同時值得抽審"),
            ("rule_basis", "STRING", "先驗命中哪條：definition / term_hint / intel_tag / default"),
            ("app", "STRING", "TV / NB / MNT / ALL（中立層 NULL → ALL；" + _YAML + " app，Stage B 按此分節）"),
            ("customer", "STRING", "正規化客戶（複製自中立層）"),
            ("term", "STRING", "主題短語（Stage A 產；" + _YAML + " term）"),
            ("summary", "STRING", "一句乾淨版（複製自中立層 summary）"),
            ("quote", "STRING", "逐字原文（複製自中立層 quote，已 token 化）；Stage A 不得改寫"),
            ("date", "DATE", "原始訊號日期（= 中立層 event_date）"),
            ("source", "STRING", "場合＋段：subject_clean ＋ segment_date ＋ speaker_side（例「acer MNT Monthly Meeting 2026-08-13 — customer」）"),
            ("tags", "ARRAY<STRING>", "topic_tags ∪ free_tags ∪ customer（" + _YAML + " tags）"),
            ("as_of", "STRING", "有效基準 yyyy-MM 或 yyyy-Qn；intel 必填（由 event_month 推），semantic / event NULL"),
            ("dup_of", "STRING", "命中 semantic_record.md 既有條目 → SEM-xxx（補證據不新增）；PM / Stage B 填，Stage A 目前留 NULL"),
            ("dup_key", "STRING", "同文重複：quote 正規化後與更早的條目相同 → 最早那條的 item_key（v_meeting_items_sem_cand.dup_key）；分類由最早那條複製，不另呼叫 LLM（D30）"),
            ("route_hint", "STRING", "建議去向：semantic → 投影代號（SemanticLayerPlan §1.3）；intel → background_intel:<app>；event → NULL"),
            ("extract_note", "STRING", "Stage A 的判類理由、內外訊號矛盾、交叉引用"),
            ("verified", "STRING", "PM 填：y / n / NULL = 未審。**Stage A MERGE 不覆蓋**"),
            ("review_note", "STRING", "PM 專屬：措辭修正、衝突說明、改 category 理由。**Stage A MERGE 不覆蓋**"),
            ("sem_id", "STRING", "定案後 Stage B 發的 SEM id（回填）；未定案 NULL"),
            ("statement_type", "STRING", "複製自中立層（查詢 / 抽審用）"),
            ("topic_tag_primary", "STRING", "複製自中立層"),
            ("content_type", "STRING", "來源段類型（inline_minutes / thread_reply…）；§8.3 紀要降權用"),
            ("sensitivity", "STRING", "S0 / S1 / S2（複製自中立層）；Stage B 彙整與對外都要看"),
            ("confidence", "DOUBLE", "Stage A 自評 0–1，只做排序與抽審"),
            ("active", "BOOLEAN", "中立層目前仍有此 item_key 的有效條目；重萃後 quote 變了的舊 key 標 false，不刪"),
            ("stage_a_version", "STRING", "lineage：Stage A prompt / 規則版本"),
            ("model", "STRING", "lineage：模型名；純規則寫入為 rule"),
            ("classified_at", "TIMESTAMP", "Stage A 判類時間"),
            ("updated_at", "TIMESTAMP", "最近一次 MERGE 時間"),
        ],
    },
    "b_meeting_migration_log": {
        "comment": "會議萃取 migration 執行紀錄（只增不改）",
        "key": ["slug", "run_at"],
        "columns": [
            ("slug", "STRING", "migration 流水號＋名稱（例 002_meeting_neutral_tables）"),
            ("run_at", "TIMESTAMP", "執行時間"),
            ("run_by", "STRING", "current_user()"),
            ("result", "STRING", "JSON：各物件 created / exists / replaced"),
            ("note", "STRING", "備註"),
        ],
    },
}


In [ ]:
# [c05] view_specs
# 正本。select = [(expr, 輸出欄名, COMMENT)]；body(fq) 回傳 FROM 起的 SQL，fq(表名) 給完整名稱。
# 兩個 View：
#   v_meeting_items_sem_cand = Stage A 的輸入（候選規則用 SQL 先算；中立層有效條目 ⋈ 清洗表 content_type ⋈ 鏡頭表已判狀態）
#   v_meeting_items_sem_q    = 鏡頭表的消費介面（只留 active，陣列欄字串版）
# 任何 View 不出現姓名 / email（[c03] validate_specs 會擋）。
_ACTIVE = "i.extract_status = 'ok' AND i.superseded = false"
_INTEL_TAGS_SQL = ", ".join(f"'{t}'" for t in INTEL_TAGS)
_INTEL_ST_SQL = ", ".join(f"'{t}'" for t in INTEL_STATEMENT_TYPES)
_LOW_CT_SQL = ", ".join(f"'{t}'" for t in LOW_WEIGHT_CONTENT_TYPES)
_WEEK_EXPR = "concat(extract(YEAROFWEEK FROM i.extracted_at), '-W', lpad(weekofyear(i.extracted_at), 2, '0'))"
_RULE_BASIS_EXPR = f"""CASE
        WHEN i.statement_type = 'definition' THEN 'definition'
        WHEN size(i.terms_mentioned) > 0 AND i.quote RLIKE '{SEMANTIC_HINT_RE}' THEN 'term_hint'
        WHEN i.statement_type IN ({_INTEL_ST_SQL}) AND i.topic_tag_primary IN ({_INTEL_TAGS_SQL}) THEN 'intel_tag'
        ELSE 'default' END"""
_CATEGORY_RULE_EXPR = f"""CASE
        WHEN i.statement_type = 'definition' THEN 'semantic'
        WHEN size(i.terms_mentioned) > 0 AND i.quote RLIKE '{SEMANTIC_HINT_RE}' THEN 'semantic'
        WHEN i.statement_type IN ({_INTEL_ST_SQL}) AND i.topic_tag_primary IN ({_INTEL_TAGS_SQL}) THEN 'intel'
        ELSE 'event' END"""
# 同文重複（D30）：quote 只留中英數字、小寫；≥ DUP_MIN_CHARS 才比；同 unit 同文 → 最早（段日期、item_key）那條的 item_key，自己是最早則 NULL。
_QN_EXPR = "lower(regexp_replace(i.quote, '[^0-9A-Za-z一-鿿]', ''))"
_DUP_KEY_EXPR = (f"CASE WHEN length({_QN_EXPR}) >= {DUP_MIN_CHARS} THEN nullif(first_value(i.item_key) OVER ("
                 f"PARTITION BY i.unit, {_QN_EXPR} ORDER BY coalesce(c.segment_date, i.event_date), i.item_key), i.item_key) END")
_SOURCE_EXPR = "concat_ws(' — ', concat_ws(' ', i.subject_clean, cast(c.segment_date AS STRING)), i.speaker_side)"
_TAGS_EXPR = ("array_distinct(filter(concat(coalesce(i.topic_tags, array()), coalesce(i.free_tags, array()), "
              "array(i.customer)), x -> x IS NOT NULL AND x <> ''))")


def _cand_body(fq):
    return f"""FROM {fq('g_meeting_items_bu')} i
LEFT JOIN {fq('s_meeting_record_clean_bu')} c
  ON c.pk_seq = i.pk_seq AND c.segment_no = i.segment_no AND c.sub_no = i.sub_no
LEFT JOIN {fq('g_meeting_items_sem_bu')} s ON s.item_key = i.item_key
WHERE {_ACTIVE}"""


def _sem_q_body(fq):
    return f"""FROM {fq('g_meeting_items_sem_bu')} s
WHERE s.active = true"""


VIEWS = {
    "v_meeting_items_sem_cand": {
        "comment": "語意鏡頭 Stage A 的輸入（BU）：中立層有效條目加 SQL 先驗 category_rule、source、tags、week，並標示鏡頭表是否已判過。LLM 只判 already_classified = false 的列。不含姓名 email。正本：003 [c05]。",
        "body": _cand_body,
        "select": [
            ("i.item_id", "item_id", "中立條目 id"),
            ("i.item_key", "item_key", "承接鍵"),
            ("i.pk_seq", "pk_seq", "來源信 pk_seq"),
            ("i.segment_no", "segment_no", "來源段序"),
            ("i.sub_no", "sub_no", "來源子段序"),
            ("i.unit", "unit", "單位代碼"),
            (_WEEK_EXPR, "week", "YYYY-Www，由 extracted_at 推"),
            (_CATEGORY_RULE_EXPR, "category_rule", "SQL 先驗候選：semantic / intel / event（§8.3 候選規則）"),
            (_RULE_BASIS_EXPR, "rule_basis", "命中哪條：definition / term_hint / intel_tag / default"),
            ("coalesce(i.app, 'ALL')", "app", "TV / NB / MNT / ALL（NULL → ALL）"),
            ("i.customer", "customer", "正規化客戶"),
            ("i.customer_raw", "customer_raw", "素材原名"),
            ("i.statement_type", "statement_type", "陳述類型"),
            ("i.topic_tag_primary", "topic_tag_primary", "主標籤"),
            ("array_join(i.topic_tags, ', ')", "topic_tags_str", "主題標籤，逗號字串"),
            ("array_join(i.free_tags, ', ')", "free_tags_str", "自由關鍵字，逗號字串"),
            ("array_join(i.terms_mentioned, ', ')", "terms_str", "命中術語，逗號字串"),
            ("i.summary", "summary", "一句摘要"),
            ("i.quote", "quote", "逐字原文（已 token 化）；Stage A 逐字抄這欄，不改寫"),
            ("i.event_date", "date", "原始訊號日期（= event_date）"),
            ("i.event_month", "event_month", "yyyy-MM；intel 的 as_of 由此推"),
            (_SOURCE_EXPR, "source", "場合＋段日期＋發言方"),
            (_TAGS_EXPR, "tags", "topic_tags ∪ free_tags ∪ customer"),
            ("coalesce(i.speaker_side, 'unknown')", "speaker_side", "發言方"),
            ("i.subject_clean", "subject_clean", "會議名稱"),
            ("c.segment_date", "segment_date", "段日期"),
            ("c.content_type", "content_type", "來源段類型"),
            (f"CASE WHEN c.content_type IN ({_LOW_CT_SQL}) THEN 'low' ELSE 'normal' END", "weight_hint",
             "§8.3：已書面化的紀要段 low，其餘 normal"),
            ("i.sensitivity", "sensitivity", "S0 / S1 / S2"),
            ("i.confidence", "confidence", "L1 自評信心"),
            ("s.item_key IS NOT NULL", "already_classified", "鏡頭表已有此 item_key"),
            ("s.category", "sem_category", "鏡頭表已判的 category（未判 NULL）"),
            ("s.verified", "sem_verified", "鏡頭表 PM 審核狀態（未判 NULL）"),
            ("s.active", "sem_active", "鏡頭表的 active（未判 NULL）；false = 曾被停用、中立層又出現 → Stage A 重判"),
            ("s.stage_a_version", "sem_stage_a_version", "鏡頭表記的 Stage A 版本（未判 NULL）；reclassify 用"),
            (_DUP_KEY_EXPR, "dup_key", "同文重複：最早那條的 item_key；NULL = 自己是最早或沒有重複（D30）"),
            ("i.prompt_version", "prompt_version", "lineage：L1 prompt 版本"),
            ("i.cleaner_version", "cleaner_version", "lineage：清洗版本"),
            ("i.extracted_at", "extracted_at", "L1 萃取時間"),
        ],
    },
    "v_meeting_items_sem_q": {
        "comment": "語意鏡頭消費介面（BU）：只留 active 列，陣列欄附逗號字串版。PM 審核、Stage B 彙整、Genie 用。不含姓名 email。正本：003 [c05]。",
        "body": _sem_q_body,
        "select": [
            ("s.item_key", "item_key", "承接鍵"),
            ("s.item_id", "item_id", "目前對應的中立條目 id"),
            ("s.pk_seq", "pk_seq", "來源信 pk_seq"),
            ("s.segment_no", "segment_no", "來源段序"),
            ("s.unit", "unit", "單位代碼"),
            ("s.week", "week", "YYYY-Www"),
            ("s.category", "category", "semantic / intel / event"),
            ("s.category_rule", "category_rule", "SQL 先驗候選"),
            ("s.category <> s.category_rule", "rule_disagree", "LLM 判定與先驗不同（抽審優先）"),
            ("s.app", "app", "TV / NB / MNT / ALL"),
            ("s.customer", "customer", "正規化客戶"),
            ("s.term", "term", "主題短語"),
            ("s.summary", "summary", "一句摘要"),
            ("s.quote", "quote", "逐字原文（已 token 化）"),
            ("s.date", "date", "原始訊號日期"),
            ("s.source", "source", "場合＋段日期＋發言方"),
            ("s.tags", "tags", "標籤陣列"),
            ("array_join(s.tags, ', ')", "tags_str", "標籤，逗號字串"),
            ("s.as_of", "as_of", "有效基準（intel）"),
            ("s.dup_of", "dup_of", "既有 SEM id"),
            ("s.dup_key", "dup_key", "同文重複：最早那條的 item_key"),
            ("s.route_hint", "route_hint", "建議去向"),
            ("s.extract_note", "extract_note", "Stage A 判類理由"),
            ("s.verified", "verified", "PM：y / n / NULL"),
            ("s.review_note", "review_note", "PM 備註"),
            ("s.sem_id", "sem_id", "定案 SEM id"),
            ("s.statement_type", "statement_type", "陳述類型"),
            ("s.content_type", "content_type", "來源段類型"),
            ("s.sensitivity", "sensitivity", "S0 / S1 / S2"),
            ("s.confidence", "confidence", "Stage A 自評"),
            ("s.stage_a_version", "stage_a_version", "lineage"),
            ("s.model", "model", "lineage"),
            ("s.classified_at", "classified_at", "判類時間"),
        ],
    },
}


In [ ]:
# [c10] build_ddl
errors = validate_specs(TABLES, VIEWS)
for e in errors:
    print("ERROR", e)
assert not errors, f"spec 驗證失敗 {len(errors)} 筆，修 [c04] / [c05] 後重跑"


def fq(name: str) -> str:
    return f"{catalog}.{schema}.{name}"


table_ddl = {t: build_create_table(fq(t), s) for t, s in TABLES.items()}
view_ddl = {v: build_create_view(fq(v), s, fq) for v, s in VIEWS.items()}
existing = {t: spark.catalog.tableExists(fq(t)) for t in list(TABLES) + list(VIEWS)}
print("既有物件：", {k: v for k, v in existing.items() if v} or "無")
print("將建立：", [k for k, v in existing.items() if not v])
if dry_run:
    print("dry_run = true：只印 DDL，不建任何物件。確認後把 dry_run 改 false 重跑。\n")
    for ddl in list(table_ddl.values()) + list(view_ddl.values()):
        print(ddl, end=";\n\n")

In [ ]:
# [c20] create_tables
# 一次性 DDL；IF NOT EXISTS 讓重跑無害。既有表的欄位差異由 [c30] 報告，不在這裡 ALTER。
if not dry_run:
    for t, ddl in table_ddl.items():
        spark.sql(ddl)
        print(("exists " if existing[t] else "created") + " " + fq(t))

In [ ]:
# [c21] create_views
# View 無狀態：CREATE OR REPLACE，改 [c05] 後重跑即生效。必須在 [c20] 之後（View 引用四張表）。
if not dry_run:
    for v, ddl in view_ddl.items():
        spark.sql(ddl)
        print(("replaced" if existing[v] else "created ") + " " + fq(v))

In [ ]:
# [c22] write_log
# 每次正式執行（dry_run = false）寫一列 b_meeting_migration_log；只 INSERT，不改舊列。
if not dry_run:
    result = {
        "tables": {t: ("exists" if existing[t] else "created") for t in TABLES},
        "views": {v: ("replaced" if existing[v] else "created") for v in VIEWS},
    }
    spark.sql(
        f"INSERT INTO {fq('b_meeting_migration_log')} "
        "SELECT :slug, current_timestamp(), current_user(), :result, :note",
        args={"slug": SLUG, "result": json.dumps(result, ensure_ascii=False), "note": None},
    )
    print("logged", result)

In [ ]:
# [c30] check_target
# 比對實際表欄位 vs spec：缺欄（表早已存在、spec 後來加欄）→ 印補欄語句並 fail；型別不同 → 警告。
# View：能查、輸出欄 = spec、不含 S3 欄。
if not dry_run:
    problems = []
    for t, spec in TABLES.items():
        actual = {f.name: f.dataType.simpleString() for f in spark.table(fq(t)).schema.fields}
        missing = [n for n, _, _ in spec["columns"] if n not in actual]
        if missing:
            problems.append(f"{t} 缺欄 {missing}。補欄（開新流水號執行）：\n  {add_columns_sql(fq(t), spec, missing)}")
        for n, typ, _ in spec["columns"]:
            if n in actual and actual[n] != typ.lower():
                print(f"WARN {t}.{n} 型別 spec={typ.lower()} actual={actual[n]}")
        extra = [n for n in actual if n not in {c[0] for c in spec["columns"]}]
        if extra:
            print(f"INFO {t} 有 spec 以外的欄（不處理）：{extra}")
    for v, spec in VIEWS.items():
        cols = spark.table(fq(v)).columns
        want = [a for _, a, _ in spec["select"]]
        if cols != want:
            problems.append(f"{v} 輸出欄與 spec 不同：{cols}")
        leak = sorted(S3_COLS & set(cols))
        if leak:
            problems.append(f"{v} 暴露 S3 欄：{leak}")
        spark.table(fq(v)).limit(1).collect()  # 能執行（表空也要能跑）
    for p in problems:
        print("PROBLEM", p)
    assert not problems, f"{len(problems)} 個問題"
    print("tables:", {t: len(spark.table(fq(t)).columns) for t in TABLES})
    print("views :", {v: len(spark.table(fq(v)).columns) for v in VIEWS})